# N036 · 扫描线与事件排序

**目标：** 用事件增减维护时间或空间中的活跃对象。

**先修：** N034, N035。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 起止事件；同坐标次序；离线处理；最大重叠；面积扩展。

**配套讲解来源：** [同名逐章意见](../../comment/036_sweep_line_events.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的核心问题是：**给出一堆区间，问"数轴上任何一个时刻，最多同时有几个区间在场"**（叫最大重叠），以及由它派生的问题：**要把这些区间分成若干组、每组内部互不重叠，最少要分几组**。区间可以是会议时间段、班车占用的路段、内存分配的地址段——只要每个对象有"开始"和"结束"两个端点，就可以问"峰值时刻同时有几个"。

先看一个具体到数字的例子（数据来自 notebook 的 “自动测试”部分）：

- 输入：`intervals = [[1,10],[2,9],[3,8]]`，求 `min_groups`
- 输出：`3`
- 为什么是 3？我们取时刻 5 检查：区间 [1,10] 覆盖 5，区间 [2,9] 覆盖 5，区间 [3,8] 也覆盖 5。三个区间在时刻 5 全部在场，两两重叠，所以任何两个都不能分进同一组，组数不可能少于 3；反过来每个区间单独一组、共 3 组一定可行，于是答案恰好是 3。这个"3"正是最大重叠数：峰值时刻有 3 个区间同时在场。

再看一个更小的例子，说明"区间语义"会直接改变答案：

- 输入：`[[1,2],[2,3]]`，求最大重叠
- 按闭区间（两端都算）解释：输出 `2`。因为闭区间 [1,2] 和 [2,3] 在点 2 处碰面，此刻两个都在场。
- 按半开区间（含左端不含右端）解释：输出 `1`。因为 [1,2) 在点 2 已经退场、[2,3) 才刚进场，两者擦肩而过，任何时刻都没有同时在场。

同一个输入、两种语义、两个答案。这就是 “思路推导”部分 那句"端点顺序不是实现细节，而是区间语义的一部分"的含义。

## 2. 基础知识：先读懂这些词

- **扫描线（sweep line）**：想象一条竖线（或一个时间指针）从左到右扫过所有区间端点。我们不需要逐点检查数轴，只需在"有事情发生的坐标"（也就是端点处）停下来更新状态。把连续问题化成一串离散事件，这套思路就叫扫描线。
- **事件（event）**：每个区间 [l, r] 被拆成两个事件：在 l 处发生"开始事件"，记作 (l, +1)；在 r 处发生"结束事件"，记作 (r, -1)。+1 的意思是活动数加一，-1 的意思是活动数减一。
- **活动数（active count）**：扫描线停在某个坐标时，"已经开始但尚未结束"的区间个数。相邻两个事件之间没有任何端点，所以活动数在这段空档里保持不变——这正是我们只需要在事件处更新它的原因。
- **最大重叠（max overlap）**：活动数在整条扫描过程中的最大值，也就是"最多同时在场"的峰值。
- **闭区间与半开区间**：闭区间 [l, r] 两个端点都算在场，所以在 l 和 r 处它都覆盖；半开区间 [l, r) 只含左端，r 一到就退场。特别地，半开区间 [2,2) 是空区间，不覆盖任何点，本章代码会直接跳过它，而闭区间 [2,2] 是一个合法的"点区间"。
- **同坐标事件次序**：当"结束事件"和"开始事件"撞在同一个坐标上时，谁先处理？闭区间要求"先加开始、再减结束"，这样共享端点的两个区间会短暂同场（符合闭区间定义）；半开区间正相反，要求"先减结束、再加开始"。次序排错，答案就错，这不是小细节。
- **离线处理（offline processing）**：所有区间在开始时就已经全部拿到手，所以我们允许先排序、再一次性扫描，不需要应付"数据边到边问"的在线场景。
- **最少分组（minimum groups）**：把区间分配到若干组，要求同组内任何两个区间互不重叠，问最少组数。本章的结论是：它恰好等于闭区间语义下的最大重叠。

## 3. 思路推导：从小例子开始

- **Step 1：把每个区间拆成两个事件。** 区间 [l, r] 变成 (l, +1) 和 (r, -1)。拆完之后我们手里只剩一串事件，区间本身可以丢掉了。
- **Step 2：排序事件。** 第一关键字是坐标，从小到大；同一坐标上的多个事件再按第二关键字排队：闭区间让 +1 排在 -1 前面（先加后减），半开区间让 -1 排在 +1 前面（先减后加）。
- **Step 3：扫描累计。** 从左到右处理事件，维护两个变量：active（当前活动数）和 best（历史峰值）。每处理一个事件就把增量加进 active，再用 best 记录 active 的新高。
- **Step 4：读出答案。** 事件全部处理完后，best 就是最大重叠。
- **Step 5：最少分组转调。** 求 min_groups 时直接复用"最少组数 = 闭区间最大重叠"这个结论，所以代码只需一行转调 `max_overlap(intervals, closed=True)`。

**手算演示一（闭区间，输入 `[[1,2],[2,3]]`）：**

拆出的原始事件是 (1,+1), (2,-1), (2,+1), (3,-1)。按闭区间规则排序（同坐标 +1 在前）后依次处理：

| 事件 | active | best |
|---|---|---|
| (1,+1) | 1 | 1 |
| (2,+1) | 2 | 2 |
| (2,-1) | 1 | 2 |
| (3,-1) | 0 | 2 |

最终 best = 2。注意坐标 2 上"先加后减"让活动数冲到 2，这正是闭区间"共享端点算重叠"的体现。

**手算演示二（半开区间，同一输入）：** 排序规则反过来（同坐标 -1 在前），事件序列变成 (1,+1), (2,-1), (2,+1), (3,-1)，活动数依次是 1, 0, 1, 0，best = 1。同一输入换个语义，峰值就少了一个。

**手算演示三（“运行示例”部分 的小实例 `a=[[1,2],[2,3],[2,2]]`）：** 这就是 notebook 用 show_table 展示的真实数据。闭区间时 [2,2] 是合法的点区间，它贡献 (2,+1) 和 (2,-1) 两个事件。全部事件按闭区间规则排序后是 (1,+1), (2,+1), (2,+1), (2,-1), (2,-1), (3,-1)，扫描过程正是 “运行示例”部分 第二张表的内容：

| 坐标 | 事件增量 | 活动区间数 |
|---|---|---|
| 1 | +1 | 1 |
| 2 | +1 | 2 |
| 2 | +1 | 3 |
| 2 | -1 | 2 |
| 2 | -1 | 1 |
| 3 | -1 | 0 |

峰值 3 出现在坐标 2：此刻 [1,2] 还没走（闭区间含右端点）、[2,3] 刚到、[2,2] 恰好是一个点，三者同场。而按半开解释，[2,2) 是空区间被跳过，再叠加"先减后加"，峰值只剩 1。“运行示例”部分 第一张表给出的正是这一对结果：闭区间 3、半开区间 1。你可以在纸上先推一遍，再运行 “运行示例”部分 对照。

## 4. 核心代码：max_overlap

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_overlap(intervals, closed=True):
    events = []
    for left, right in intervals:
        if left > right:
            raise ValueError('reversed interval')
        if not closed and left == right:
            continue
        events.append((left, 1))
        events.append((right, -1))
    events.sort(key=lambda event: (event[0], -event[1] if closed else event[1]))
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best
```

### 逐段读懂代码

### 函数 1：`max_overlap` —— 求最大重叠

```python
def max_overlap(intervals, closed=True):
    events = []
    for left, right in intervals:
        if left > right:
            raise ValueError('reversed interval')
        if not closed and left == right:
            continue
        events.append((left, 1))
        events.append((right, -1))
    events.sort(key=lambda event: (event[0], -event[1] if closed else event[1]))
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best
```

- `def max_overlap(intervals,closed=True)`：参数 closed 默认 True，也就是默认按闭区间处理；调用方想按半开解释，就显式传 `closed=False`。
- `if left>right: raise ValueError('reversed interval')`：这行是输入守卫，挡住"左右端点写反"的坏数据。区间语义要求起点不超过终点，反着写的数据与其悄悄算出一个错误答案，不如当场报错。
- `if not closed and left==right: continue`：半开区间 [x,x) 是空区间，不覆盖任何点，拆事件时直接跳过（continue 的意思是"跳过本次循环剩余部分，直接处理下一个区间"）。闭区间则不跳过，因为 [x,x] 还覆盖着点 x。
- `events.append((left,1)); events.append((right,-1))`：把一个区间拆成"开始 +1"和"结束 -1"两个事件。一行里用分号写两个语句，是紧凑写法，效果和分两行完全一样。
- `events.sort(key=lambda event:(event[0],-event[1] if closed else event[1]))`：这是全章最关键的一行。排序的第一优先级是坐标 event[0]；第二优先级是一个三元表达式——闭区间时取 `-event[1]`（+1 被变成 -1，于是在数值上排到 -1 前面，即"开始事件"排前），半开时取 `event[1]`（-1 排前，即"结束事件"排前）。一行代码承载了"闭区间先加后减、半开先减后加"这两条语义规则。
- `active=best=0`：链式赋值，active 和 best 都从 0 开始。空输入时循环一次都不进，自然返回 0。
- `for _,delta in events: active+=delta; best=max(best,active)`：扫描阶段。变量名 `_` 是 Python 惯例，表示"这一列的值我不关心了"，这里坐标已经用完，只关心增量 delta。每一步把 delta 加进 active，再刷新 best。
- `return best`：返回峰值，就是最大重叠。

### 函数 2：`min_groups` —— 最少分组数

```python
def min_groups(intervals):
    return max_overlap(intervals, closed=True)
```

整个函数只有一行：最少组数就是闭区间语义下的最大重叠。为什么敢这样一行转调，第五节给出两个方向的完整论证。

## 5. 分段实现：按顺序运行

**本章接口：** `max_overlap(intervals, closed)`、`min_groups(intervals)`

### max_overlap

In [1]:
def max_overlap(intervals, closed=True):
    events = []
    for left, right in intervals:
        if left > right:
            raise ValueError('reversed interval')
        if not closed and left == right:
            continue
        events.append((left, 1))
        events.append((right, -1))
    events.sort(key=lambda event: (event[0], -event[1] if closed else event[1]))
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best

### min_groups

In [2]:
def min_groups(intervals):
    return max_overlap(intervals, closed=True)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[[1,2],[2,3],[2,2]]
show_table(['区间语义','最大重叠'],[('闭区间',max_overlap(a,True)),('半开区间',max_overlap(a,False))])
events=sorted([(x,d) for l,r in a for x,d in [(l,1),(r,-1)]],key=lambda z:(z[0],-z[1]))
rows=[]; active=0
for x,d in events:
    active+=d; rows.append((x,d,active))
show_table(['坐标','事件增量','活动区间数'],rows)

区间语义,最大重叠
闭区间,3
半开区间,1


坐标,事件增量,活动区间数
1,1,1
2,1,2
2,1,3
2,-1,2
2,-1,1
3,-1,0


## 7. 正确性、适用条件与复杂度

活动数在相邻事件间不变，只需检查事件位置的合法中间状态。达到最大重叠的那一刻所有区间必须分组不同；按开始时间复用最早空出的组不会超过同时活动数。

**代价：** O(n log n) 排序时间、O(n) 事件空间。

### 展开理解

**为什么 max_overlap 算出来的就是最大重叠？** 我们用两句完整的话说清楚。第一句：活动数只在端点处发生变化，因为两个相邻端点之间没有任何区间开始或结束，活动数在那段空档里纹丝不动；所以"全程所有位置的活动数"的最大值，一定在某个事件刚被处理完的时刻取得，我们只需检查每个事件处理完的活动数，就不会漏掉峰值。第二句：排序的第二个关键字保证同坐标事件按区间语义排队——闭区间先加后减，两个共享端点的区间就会短暂同场，这符合闭区间的定义；半开先减后加，前一个区间先退场，这符合半开的定义。一句话总结：我们检查的位置足够密（不会漏掉峰值），每个位置上的数值又符合语义（不会算错），两者合起来结果必然正确。

**为什么 min_groups 就等于 max_overlap？** 我们从两个方向把答案夹住。方向一（组数不可能更少）：设想活动数达到峰值的那个时刻，有 k 个区间同时在场；这 k 个区间两两重叠，所以它们必须分属 k 个不同的组，于是任何分组方案的组数都至少是 k，也就是至少等于最大重叠。方向二（这么多个组一定够用）：我们按开始时间从早到晚处理区间，每次把当前区间放进"最早空出来的那个组"——这个贪心过程恰好就是扫描线本身，任何时刻正在占用的组数都不会超过当时的活动数，而活动数的峰值就是最大重叠，所以最大重叠个组一定够。下界与上界在同一个数字上咬合，答案就唯一了。

顺带把"按时间复用最早空出的组"落到数字上（输入还是 `[[1,10],[2,9],[3,8]]`）。我们按开始时间依次处理：[1,10] 进组 1，该组要占用到时刻 10；[2,9] 来了，最早能空出的是组 1（10 时刻），10 > 2 说明还没空，只能开组 2，占用到 9；[3,8] 来了，最早能空出的换成组 2（9 时刻），9 > 3 仍没空，再开组 3。最终 3 个组，恰好等于最大重叠。你把这三步和扫描线的 active 起伏对照，会发现它们就是同一个过程的两张皮。

**复杂度是多少？** 拆事件是 O(n)，排序是 O(n log n)，扫一遍是 O(n)，总体时间 O(n log n)；事件共 2n 条，空间 O(n)。拿具体数字感受一下：n = 10⁵ 个区间时，排序大约要做 10⁵ × 17 ≈ 170 万次比较，再加上 20 万次事件扫描，普通电脑瞬间完成。对比"对每个候选坐标数一遍所有区间"的暴力做法（O(n²) 量级），这是质的飞跃。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解读：

- `assert max_overlap([[1,2],[2,3]],True)==2`：**正常值测试（闭区间共享端点）**。它验证"先加后减"确实让在点 2 碰头的两个闭区间被算作重叠，预期值 2 来自第三节手算演示一。
- `assert max_overlap([[1,2],[2,3]],False)==1`：**正常值测试（半开语义）**。同样的输入换个语义，答案应降为 1。这条专门抓"同坐标次序写反"的 bug——如果半开实现误用了先加后减，这里会算出 2 而挂掉。
- `assert min_groups([[1,10],[2,9],[3,8]])==3`：**正常值测试（最少分组）**。三个区间在时刻 5 全部在场，谁也不能与谁同组，所以是 3，同时验证了"最少组数 = 最大重叠"这条转调关系。
- `assert max_overlap([],True)==0`：**边界值测试（空输入）**。没有任何区间时峰值自然是 0，同时确认空输入不会让代码崩溃。
- 随机交叉验证块（`rng=Random(36)`，循环 200 次）：**交叉验证测试（用暴力解当裁判）**。每次随机生成 5 个区间（先取两个 0 到 4 的随机整数、排序后当左右端点，保证 left 不大于 right），然后对每个检查点 x 用笨办法直接数"有多少区间覆盖 x"：闭区间用条件 `l<=x<=r`，半开用 `l<=x<r`；检查点取 0, 0.5, 1, 1.5, …, 4（代码是 `i/2 for i in range(9)`）。笨办法在所有检查点上取最大值，必须与 max_overlap 的返回值一致。因为端点全是整数，活动数只会在整数坐标处跳变，半步长的检查点足以同时覆盖"端点刚过"和"端点之间"两种状态。

归档一下这批断言的类型分布：

| 断言 | 类型 | 它防的是哪类 bug |
|---|---|---|
| `[[1,2],[2,3]],True → 2` | 正常值（闭区间共享端点） | 闭区间漏算共享点重叠 |
| `[[1,2],[2,3]],False → 1` | 正常值（半开语义） | 同坐标次序写反 |
| `min_groups([[1,10],[2,9],[3,8]]) → 3` | 正常值（分组） | "峰值即组数"的转调错 |
| `max_overlap([],True) → 0` | 边界（空输入） | 空列表崩溃 |
| 随机 200 次对拍暴力 | 交叉验证 | 主逻辑与语义细节的整体错误 |

这批测试的编排思路值得学走：先用两条断言钉死"手算得出来的语义差异"，再用一条空输入守住底线，最后拿随机暴力把两种语义各刷 200 遍。

In [4]:
assert max_overlap([[1, 2], [2, 3]], True) == 2

assert max_overlap([[1, 2], [2, 3]], False) == 1

assert min_groups([[1, 10], [2, 9], [3, 8]]) == 3

assert max_overlap([], True) == 0

from random import Random

rng = Random(36)

for _ in range(200):
    a = [sorted([rng.randrange(5), rng.randrange(5)]) for _ in range(5)]
    points = [i / 2 for i in range(9)]
    assert max_overlap(a, True) == max((sum((l <= x <= r for l, r in a)) for x in points))
    assert max_overlap(a, False) == max((sum((l <= x < r for l, r in a)) for x in points))

print('N036: 所有本章断言通过')

N036: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 分别实现闭区间和半开区间扫描。

**练习 2：** 说明同坐标事件顺序改变答案。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（分别实现闭区间和半开区间扫描）**：提示你别依赖那一行三元表达式，而是写两个独立函数，逼自己把两种语义的差别讲清楚。边界至少考虑三种情况：空输入（应返回 0）、单个点区间 [x,x]（闭区间答案 1、半开应跳过）、以及多个端点挤在同一坐标的输入。手算示例可以直接用 [[1,2],[2,3]]：闭区间答案 2、半开答案 1；你先在纸上写出两种事件序列和 active 的起伏，再运行代码对答案。
- **练习 2（说明同坐标事件顺序会改变答案）**：核心素材还是 [[1,2],[2,3]]。建议做一个对照实验：输入不动，把排序第二关键字强行翻转（例如闭区间改成先减后加），观察答案从 2 掉到 1。然后解释原因：坐标 2 处"结束 [1,2]"和"开始 [2,3]"谁先被处理，决定了这两个区间是否同场，而这正是两种区间语义的分水岭。写边界时别忘了点区间 [2,2]：半开语义下它根本不该贡献事件。验证可以用本章接口或自己写的小暴力（对每个点直接数覆盖的区间个数）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def max_overlap(intervals, closed=True):
    events = []
    for left, right in intervals:
        if left > right:
            raise ValueError('reversed interval')
        if not closed and left == right:
            continue
        events.append((left, 1))
        events.append((right, -1))
    events.sort(key=lambda event: (event[0], -event[1] if closed else event[1]))
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best

def min_groups(intervals):
    return max_overlap(intervals, closed=True)

# 示例与回归测试
assert max_overlap([[1, 2], [2, 3]], True) == 2

assert max_overlap([[1, 2], [2, 3]], False) == 1

assert min_groups([[1, 10], [2, 9], [3, 8]]) == 3

assert max_overlap([], True) == 0

from random import Random

rng = Random(36)

for _ in range(200):
    a = [sorted([rng.randrange(5), rng.randrange(5)]) for _ in range(5)]
    points = [i / 2 for i in range(9)]
    assert max_overlap(a, True) == max((sum((l <= x <= r for l, r in a)) for x in points))
    assert max_overlap(a, False) == max((sum((l <= x < r for l, r in a)) for x in points))

print('N036: 所有本章断言通过')

N036: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **1094. Car Pooling（拼车）**：车沿线路行驶，每个路段有人上车（+1）有人下车（-1），问全程乘客数是否超过车辆容量。这题练的就是本章"把区间拆成 +1/-1 事件再扫描"的基本功，与 max_overlap 同构，只是把"记录峰值"换成"峰值与容量比大小"。
- **2406. Divide Intervals Into Minimum Number of Groups（区间最少分组）**：这题就是 min_groups 的原题。你把本章"组数下界是最大重叠、按时间复用最早空组可达到该下界"的两方向论证搬过去即可；工程上也常见"按左端点排序 + 最小堆维护各组最晚右端点"的写法，思路与本章贪心一致。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。